In [ ]:
import numpy as np
import pandas as pd
import math
import torch
import torch.nn as nn
from tqdm import trange

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

class NeuralNetwork(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=[5, 5, 5, 5, 5], activation='tanh'):
        super(NeuralNetwork, self).__init__()
        layers = []
        layers.append(nn.Linear(input_size, hidden_size[0]))
        layers.append(nn.Tanh())
        for i in range(len(hidden_size) - 1):
            layers.append(nn.Linear(hidden_size[i], hidden_size[i+1]))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_size[-1], output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

def auto_grad(u, x, order=1):
    if order == 1:
        return torch.autograd.grad(u, x, torch.ones_like(u), retain_graph=True, create_graph=True)[0]
    return auto_grad(auto_grad(u, x), x, order - 1)

def xavier_init(layer):
    if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

def transformed_matrix(phi, opt):
    anpha = np.deg2rad(phi)
    m, n = math.cos(anpha), math.sin(anpha)
    if abs(m) < 2.2204e-10: m = 0
    if abs(n) < 2.2204e-10: n = 0
    if opt==1:
        T = np.array([[m**2, n**2, 2*m*n], [n**2, m**2, -2*m*n], [-m*n, m*n, m**2-n**2]])
    else:
        T = np.array([[m**2, n**2, m*n], [n**2, m**2, -m*n], [-2*m*n, 2*m*n, m**2-n**2]])
    return T

def train_data(Nx, Ny, Nf):
    xu = np.linspace(-1, 1, Nx).reshape([Nx, 1])
    yu = np.linspace(-1, 1, Ny).reshape([Ny, 1])
    X, Y = np.meshgrid(xu, yu)
    Xf1 = np.vstack([X.ravel(), Y.ravel()]).T
    Xf1 = torch.tensor(Xf1, dtype=torch.float32, requires_grad=True)

    Xf2 = np.random.rand(Nf,2)*2-1
    Xf2 = torch.tensor(Xf2, dtype=torch.float32, requires_grad=True)
    return Xf1, Xf2

a, b, h = 5, 5, 1
q0 = 1e-3
E1 = 25
E2 = E1/25
G12 = 0.5*E2
mu12 = 0.25
mu21 = mu12*E2/E1

Q11 = E1/(1 - mu12*mu21)
Q12 = mu12*E2/(1 - mu12*mu21)
Q22 = E2/(1 - mu12*mu21)
Q66 = G12
Q = np.array([[Q11, Q12 , 0],[Q12 , Q22, 0],[0, 0, Q66]])

layups_list = [[0], [0, 90], [0, 90, 0], [0, 90, 0, 90]]
layup_names = ["(0°)", "(0°/90°)", "(0°/90°/0°)", "(0°/90°)_2"]

analytical_data = {
    "(0°)": [0.1308, 0.0439, 0.0007, -0.0870, -0.0112],
    "(0°/90°)": [0.3954, 0.0239, 0.0239, -0.0553, -0.0553],
    "(0°/90°/0°)": [0.1371, 0.0444, 0.0018, -0.0881, -0.0157],
    "(0°/90°)_2": [0.1761, 0.0249, 0.0249, -0.0565, -0.0565]
}

row_labels = [
    "w_0 at the center (0, 0)",
    "M_xx at the center (0, 0)",
    "M_yy at the center (0, 0)",
    "M_xx at mid-edge (-a/2, 0)",
    "M_yy at mid-edge (0, -b/2)"
]

table_rows = []
Nxb, Nyb, Nf1 = 100, 100, 10000
epochs = 10000 # Chạy 10.000 epochs để tối ưu Loss

def Energy_loss_CLPT(x, Net_w, func_w, Net_u, func_u, Net_v, func_v, A_mat, B_mat, D_mat):
    q = q0
    u = Net_u(x)*(func_u(x).view(-1,1))
    v = Net_v(x)*(func_v(x).view(-1,1))
    w = Net_w(x)*(func_w(x).view(-1,1))

    du_x = auto_grad(u, x, 1)[:,0].view(-1,1)
    du_y = auto_grad(u, x, 1)[:,1].view(-1,1)
    dv_x = auto_grad(v, x, 1)[:,0].view(-1,1)
    dv_y = auto_grad(v, x, 1)[:,1].view(-1,1)
    dw_x = auto_grad(w, x, 1)[:,0].view(-1,1)
    dw_y = auto_grad(w, x, 1)[:,1].view(-1,1)

    dw_xx = auto_grad(dw_x, x, 1)[:,0].view(-1,1)
    dw_yy = auto_grad(dw_y, x, 1)[:,1].view(-1,1)
    dw_xy = auto_grad(dw_x, x, 1)[:,1].view(-1,1)

    w_phys = w*h
    dw_x, dw_y, du_y, dv_x = dw_x*h/a, dw_y*h/b, du_y*a/b, dv_x*b/a
    dw_xx, dw_yy, dw_xy = dw_xx*h/a**2, dw_yy*h/b**2, dw_xy*h/a/b

    eps_xx = du_x + 0.5*dw_x**2
    eps_yy = dv_y + 0.5*dw_y**2
    eps_xy = 0.5*(du_y + dv_x) + 0.5*dw_y*dw_x
    k_xx, k_yy, k_xy = -dw_xx, -dw_yy, -dw_xy

    N_xx = A_mat[0,0]*eps_xx + A_mat[0,1]*eps_yy + A_mat[0,2]*2*eps_xy  + B_mat[0,0]*k_xx + B_mat[0,1]*k_yy + B_mat[0,2]*2*k_xy
    N_yy = A_mat[1,0]*eps_xx + A_mat[1,1]*eps_yy + A_mat[1,2]*2*eps_xy  + B_mat[1,0]*k_xx + B_mat[1,1]*k_yy + B_mat[1,2]*2*k_xy
    N_xy = A_mat[2,0]*eps_xx + A_mat[2,1]*eps_yy + A_mat[2,2]*2*eps_xy  + B_mat[2,0]*k_xx + B_mat[2,1]*k_yy + B_mat[2,2]*2*k_xy

    M_xx = B_mat[0,0]*eps_xx + B_mat[0,1]*eps_yy + B_mat[0,2]*2*eps_xy  + D_mat[0,0]*k_xx + D_mat[0,1]*k_yy + D_mat[0,2]*2*k_xy
    M_yy = B_mat[1,0]*eps_xx + B_mat[1,1]*eps_yy + B_mat[1,2]*2*eps_xy  + D_mat[1,0]*k_xx + D_mat[1,1]*k_yy + D_mat[1,2]*2*k_xy
    M_xy = B_mat[2,0]*eps_xx + B_mat[2,1]*eps_yy + B_mat[2,2]*2*eps_xy  + D_mat[2,0]*k_xx + D_mat[2,1]*k_yy + D_mat[2,2]*2*k_xy

    U_m = 0.5*(eps_xx*N_xx + eps_yy*N_yy + 2*eps_xy*N_xy)
    U_b = 0.5*(k_xx*M_xx + k_yy*M_yy + 2*k_xy*M_xy)
    U_e = q*w_phys

    return torch.mean(U_m), torch.mean(U_b), torch.mean(U_e)
# ==========================================================

for idx, phi in enumerate(layups_list):
    name = layup_names[idx]
    print(f"\n---> Đang huấn luyện mô hình cho tấm: {name}")

    n_layer = len(phi)
    t = h / n_layer

    Q_bar = []
    for i in range(n_layer):
        T2 = transformed_matrix(phi[i], 2)
        Q2 = T2.T @ Q @ T2
        Q_bar.append(Q2)

    z1 = np.array([((i)-n_layer/2)*t for i in range(n_layer)])
    z2 = np.array([((i+1)-n_layer/2)*t for i in range(n_layer)])

    A_mat, B_mat, D_mat = np.zeros((3,3)), np.zeros((3,3)), np.zeros((3,3))
    for i in range(n_layer):
        A_mat += Q_bar[i] * (z2[i] - z1[i])
        B_mat += Q_bar[i] * (z2[i]**2 - z1[i]**2)/2
        D_mat += Q_bar[i] * (z2[i]**3 - z1[i]**3)/3

    Net_w = NeuralNetwork(2, 1).to(device)
    Net_u = NeuralNetwork(2, 1).to(device)
    Net_v = NeuralNetwork(2, 1).to(device)
    Net_w.apply(xavier_init); Net_u.apply(xavier_init); Net_v.apply(xavier_init)

    func_w = lambda x: ((x[:,0]+1)*(x[:,0]-1)*(x[:,1]+1)*(x[:,1]-1))**2
    func_u = lambda x: ((x[:,0]+1)*(x[:,0]-1)*(x[:,1]+1)*(x[:,1]-1))
    func_v = lambda x: ((x[:,0]+1)*(x[:,0]-1)*(x[:,1]+1)*(x[:,1]-1))

    params = list(Net_w.parameters()) + list(Net_u.parameters()) + list(Net_v.parameters())
    optimizer_Adam = torch.optim.Adam(params, lr=0.001)

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer_Adam, step_size=2000, gamma=0.8)

    td = trange(epochs, dynamic_ncols=True, ncols=80)
    for epoch in td:
        if epoch < 100:
            Xf1, _ = train_data(Nxb, Nyb, Nf1)
            Xf = Xf1.to(device)
        else:
            _, Xf2 = train_data(Nxb, Nyb, Nf1)
            Xf = Xf2.to(device)

        U_m, U_b, U_e = Energy_loss_CLPT(Xf, Net_w, func_w, Net_u, func_u, Net_v, func_v, A_mat, B_mat, D_mat)
        loss = U_m + U_b - U_e
        loss.backward()
        optimizer_Adam.step()
        optimizer_Adam.zero_grad()
        scheduler.step()

        td.set_description(f"Loss:{loss:.2e}")

    # ===== LẤY KẾT QUẢ CHO TỪNG TẤM =====
    Net_w.eval().cpu(); Net_u.eval().cpu(); Net_v.eval().cpu()
    pts_eval = torch.tensor([[0.0, 0.0], [-1.0, 0.0], [0.0, -1.0]], dtype=torch.float32, requires_grad=True)

    u_eval = Net_u(pts_eval)*(func_u(pts_eval).view(-1,1))
    v_eval = Net_v(pts_eval)*(func_v(pts_eval).view(-1,1))
    w_eval = Net_w(pts_eval)*(func_w(pts_eval).view(-1,1))

    du_x_eval = auto_grad(u_eval, pts_eval, 1)[:,0].view(-1,1)
    du_y_eval = auto_grad(u_eval, pts_eval, 1)[:,1].view(-1,1)
    dv_x_eval = auto_grad(v_eval, pts_eval, 1)[:,0].view(-1,1)
    dv_y_eval = auto_grad(v_eval, pts_eval, 1)[:,1].view(-1,1)

    dw_x_eval = auto_grad(w_eval, pts_eval, 1)[:,0].view(-1,1)
    dw_y_eval = auto_grad(w_eval, pts_eval, 1)[:,1].view(-1,1)
    dw_xx_eval = auto_grad(dw_x_eval, pts_eval, 1)[:,0].view(-1,1)
    dw_yy_eval = auto_grad(dw_y_eval, pts_eval, 1)[:,1].view(-1,1)
    dw_xy_eval = auto_grad(dw_x_eval, pts_eval, 1)[:,1].view(-1,1)

    w_phys = w_eval * h
    dw_x_eval, dw_y_eval = dw_x_eval*h/a, dw_y_eval*h/b
    du_y_eval, dv_x_eval = du_y_eval*a/b, dv_x_eval*b/a
    dw_xx_eval, dw_yy_eval, dw_xy_eval = dw_xx_eval*h/a**2, dw_yy_eval*h/b**2, dw_xy_eval*h/a/b

    eps_xx_eval = du_x_eval + 0.5*dw_x_eval**2
    eps_yy_eval = dv_y_eval + 0.5*dw_y_eval**2
    eps_xy_eval = 0.5*(du_y_eval + dv_x_eval) + 0.5*dw_y_eval*dw_x_eval

    k_xx_eval, k_yy_eval, k_xy_eval = -dw_xx_eval, -dw_yy_eval, -dw_xy_eval

    M_xx_eval = B_mat[0,0]*eps_xx_eval + B_mat[0,1]*eps_yy_eval + B_mat[0,2]*2*eps_xy_eval + D_mat[0,0]*k_xx_eval + D_mat[0,1]*k_yy_eval + D_mat[0,2]*2*k_xy_eval
    M_yy_eval = B_mat[1,0]*eps_xx_eval + B_mat[1,1]*eps_yy_eval + B_mat[1,2]*2*eps_xy_eval + D_mat[1,0]*k_xx_eval + D_mat[1,1]*k_yy_eval + D_mat[1,2]*2*k_xy_eval

    W_bar_eval = w_phys.detach().numpy() * 100 * (h)**3 * E2 / (q0 * (2*a)**4)
    M_xx_bar_eval = M_xx_eval.detach().numpy() / (q0 * (2*a)**2)
    M_yy_bar_eval = M_yy_eval.detach().numpy() / (q0 * (2*a)**2)

    pinn_preds = [
        W_bar_eval[0][0], M_xx_bar_eval[0][0], M_yy_bar_eval[0][0],
        M_xx_bar_eval[1][0], M_yy_bar_eval[2][0]
    ]
    ana_sols = analytical_data[name]

    table_rows.append([f"{name} laminated plate", "", "", ""])
    for i in range(5):
        pred = pinn_preds[i]
        ana = ana_sols[i]
        error = abs(pred - ana) / abs(ana) * 100 if ana != 0 else 0
        table_rows.append([row_labels[i], f"{pred:.4f}", f"{ana:.4f}", f"{error:.2f}%"])

df_final = pd.DataFrame(table_rows, columns=["", "Predicted results", "Analytical solutions", "Error"])
print("\n" + "="*80)
print("Table 1: Comparison of predicted and analytical solutions")
print("="*80)
print(df_final.to_string(index=False))
print("="*80 + "\n")

Using device: cpu


---> Đang huấn luyện mô hình cho tấm: (0°)


Loss:-1.69e-06: 100%|██████████| 10000/10000 [13:28<00:00, 12.37it/s]



---> Đang huấn luyện mô hình cho tấm: (0°/90°)


Loss:-4.63e-06: 100%|██████████| 10000/10000 [13:45<00:00, 12.11it/s]



---> Đang huấn luyện mô hình cho tấm: (0°/90°/0°)


Loss:-8.44e-07: 100%|██████████| 10000/10000 [13:32<00:00, 12.30it/s]



---> Đang huấn luyện mô hình cho tấm: (0°/90°)_2


Loss:-2.20e-06: 100%|██████████| 10000/10000 [13:32<00:00, 12.31it/s]


Table 1: Comparison of predicted and analytical solutions
                            Predicted results Analytical solutions  Error
       (0°) laminated plate                                              
   w_0 at the center (0, 0)            0.1219               0.1308  6.81%
  M_xx at the center (0, 0)            0.0385               0.0439 12.30%
  M_yy at the center (0, 0)            0.0006               0.0007 10.84%
 M_xx at mid-edge (-a/2, 0)           -0.0882              -0.0870  1.37%
 M_yy at mid-edge (0, -b/2)           -0.0072              -0.0112 35.39%
   (0°/90°) laminated plate                                              
   w_0 at the center (0, 0)            0.3977               0.3954  0.58%
  M_xx at the center (0, 0)            0.0224               0.0239  6.14%
  M_yy at the center (0, 0)            0.0276               0.0239 15.64%
 M_xx at mid-edge (-a/2, 0)           -0.0602              -0.0553  8.94%
 M_yy at mid-edge (0, -b/2)           -0.0454        